# Дообучение `ru_core_news_sm` на russian-pii-66k

Тест из `../testing/russian_pii_66k_test_manifest.json` закреплён до обучения и совпадает с выборкой базового ноутбука. `russian_pii_66k_split_manifest.json` хранит индексы validation; train — остальные записи. Оба манифеста и SHA-256 исходного JSONL проверяются при каждом запуске.

`GIVENNAME` + `SURNAME` объединяются в `PER`, `CITY` и `STREET` переходят в `LOC`, остальные типы PII сохраняют исходные метки. Это позволяет сравнить `PER`/`LOC` с базовым тестом и отдельно оценить остальные типы. Для обучения исключаются целые записи с сущностями, границы которых не совпадают с токенами spaCy; при оценке такие сущности остаются в эталоне. Данные синтетические; результат не характеризует качество на реальных обращениях.

Запускайте с ядром окружения проекта после `venv/bin/pip install -r requirements.txt` и `venv/bin/python -m spacy download ru_core_news_sm`. Обучение полного корпуса может занять продолжительное время.

## Какие сущности распознаёт модель

В таблице перечислены все 17 исходных классов разметки. Модель обучается на 15 метках: имя и фамилия переходят в `PER`, город и улица — в `LOC`. Если части имени стоят рядом, они объединяются в одну сущность `PER`; город и улица остаются отдельными сущностями `LOC`. Примеры взяты из обучающей части синтетического датасета и показывают формат разметки, а не реальные персональные данные.

| Исходный класс | Метка модели | Что означает | Три примера |
| --- | --- | --- | --- |
| `GIVENNAME` | `PER` | Личное имя человека. | `Гремислав`; `Матвей`; `Нина` |
| `SURNAME` | `PER` | Фамилия человека. | `Гришин`; `Меркушева`; `Киселева` |
| `CITY` | `LOC` | Название населённого пункта, иногда вместе с сокращением его типа. | `д. Костомукша`; `ст. Якша`; `г. Токсово` |
| `STREET` | `LOC` | Название улицы, переулка или другого элемента уличного адреса. | `наб. Санаторная`; `ул. Кирпичная`; `пер. Тенистый` |
| `ACCOUNTNUM` | `ACCOUNTNUM` | Номер счёта, указанный в тексте. | `RU45FOMI8350305641395`; `RU59KPAD5430391171822`; `RU39NOXB1065133387262` |
| `BUILDINGNUM` | `BUILDINGNUM` | Номер дома или строения в адресе. | `2`; `131`; `3/1` |
| `CREDITCARDNUMBER` | `CREDITCARDNUMBER` | Номер платёжной карты. | `4226916697848015`; `344657871331502`; `343619399091696` |
| `DATEOFBIRTH` | `DATEOFBIRTH` | Дата рождения человека. | `19/03/1976`; `16.11.2005`; `2003-06-30` |
| `DRIVERLICENSENUM` | `DRIVERLICENSENUM` | Номер водительского удостоверения. | `78 25 496922`; `51 17 340174`; `81 78 375504` |
| `EMAIL` | `EMAIL` | Адрес электронной почты. | `gavrila1984@example.org`; `polina_1984@example.net`; `uljan1979@example.net` |
| `IDCARDNUM` | `IDCARDNUM` | Номер документа, удостоверяющего личность. | `29 37 900581`; `95 44 835911`; `14 50 520651` |
| `PASSWORD` | `PASSWORD` | Пароль для входа или доступа. | `uS9zneU2&78AF6ly`; `PAPGrM$Vi901xu`; `F%9sJm@8uS` |
| `SOCIALNUM` | `SOCIALNUM` | Номер социального страхования. | `511-615-594 07`; `192-832-764 83`; `413-953-767 24` |
| `TAXNUM` | `TAXNUM` | Налоговый идентификатор. | `659818733147`; `711305304603`; `692853045900` |
| `TELEPHONENUM` | `TELEPHONENUM` | Номер телефона. | `8 (849) 696-5328`; `+78932528809`; `87376311656` |
| `USERNAME` | `USERNAME` | Имя пользователя или логин. | `sigizmundafanasev`; `ivan_2003`; `arhip_35` |
| `ZIPCODE` | `ZIPCODE` | Почтовый индекс. | `462704`; `473829`; `595148` |

Например, соседние `Гремислав` (`GIVENNAME`) и `Гришин` (`SURNAME`) превращаются в одну сущность `Гремислав Гришин` с меткой `PER`. Исходная модель также знает `ORG`, но в этом корпусе нет обучающей разметки для организаций.

In [1]:
import hashlib
import json
import random
from collections import Counter
from pathlib import Path

import spacy
from spacy.training import Example
from spacy.util import minibatch

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "notebooks/education/russian_pii_66k_split_manifest.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Не найден корень проекта")
DATA_PATH = ROOT / "data/russian_pii_66k.jsonl"
SPLIT_PATH = ROOT / "notebooks/education/russian_pii_66k_split_manifest.json"
SPLIT = json.loads(SPLIT_PATH.read_text(encoding="utf-8"))
TEST_PATH = (SPLIT_PATH.parent / SPLIT["test_manifest"]).resolve()
TEST = json.loads(TEST_PATH.read_text(encoding="utf-8"))
if not DATA_PATH.is_file():
    raise FileNotFoundError(DATA_PATH)
digest = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
if digest != SPLIT["sha256"] or digest != TEST["sha256"]:
    raise ValueError("SHA-256 датасета не совпадает с манифестами")
if SPLIT["revision"] != TEST["revision"] or SPLIT["row_count"] != TEST["row_count"]:
    raise ValueError("Ревизия или размер набора в манифестах не совпадают")
rows = [json.loads(line) for line in DATA_PATH.open(encoding="utf-8")]
if len(rows) != SPLIT["row_count"]:
    raise ValueError("Неверное число записей")
test_ids = set(TEST["indices"])
val_ids = set(SPLIT["validation_indices"])
all_ids = set(range(len(rows)))
if (len(test_ids) != TEST["sample_size"] or len(val_ids) != SPLIT["validation_count"]
        or not test_ids.isdisjoint(val_ids) or not (test_ids | val_ids) <= all_ids):
    raise ValueError("Некорректные или пересекающиеся индексы")
train_ids = sorted(all_ids - test_ids - val_ids)
if len(train_ids) != SPLIT["train_count"] or len(test_ids) != SPLIT["test_count"]:
    raise ValueError("Размеры частей не совпадают с манифестом")
train_rows = [rows[i] for i in train_ids]
val_rows = [rows[i] for i in sorted(val_ids)]
test_rows = [rows[i] for i in sorted(test_ids)]
print(f"SHA-256: {digest}; train={len(train_rows)}, val={len(val_rows)}, test={len(test_rows)}")

SHA-256: 0b02943c2c090f448ed48549613ca412987d7ea4b35de2ff66dbaab2aecfe692; train=58187, val=6465, test=1000


In [2]:
LABEL_MAP = {"GIVENNAME": "PER", "SURNAME": "PER", "CITY": "LOC", "STREET": "LOC"}


def gold_spans(row):
    """Сопоставляет имена и адреса с метками модели, сохраняя остальные типы PII."""
    source = row["source_text"]
    spans = sorted((item["start"], item["end"], LABEL_MAP.get(item["label"], item["label"]))
                   for item in row["privacy_mask"])
    merged = []
    for start, end, label in spans:
        if (label == "PER" and merged and merged[-1][2] == "PER"
                and 0 < start - merged[-1][1] <= 3
                and source[merged[-1][1]:start].isspace()):
            merged[-1] = (merged[-1][0], end, label)
        else:
            merged.append((start, end, label))
    return merged


nlp = spacy.load("ru_core_news_sm")
if nlp.meta["version"] != "3.8.0":
    raise ValueError("Для сопоставимости нужна ru_core_news_sm 3.8.0")
labels = sorted({label for row in train_rows for _, _, label in gold_spans(row)})
for label in labels:
    nlp.get_pipe("ner").add_label(label)
print(f"spaCy {spacy.__version__}; модель {nlp.meta['version']}; метки: {labels}")

spaCy 3.8.16; модель 3.8.0; метки: ['ACCOUNTNUM', 'BUILDINGNUM', 'CREDITCARDNUMBER', 'DATEOFBIRTH', 'DRIVERLICENSENUM', 'EMAIL', 'IDCARDNUM', 'LOC', 'PASSWORD', 'PER', 'SOCIALNUM', 'TAXNUM', 'TELEPHONENUM', 'USERNAME', 'ZIPCODE']


In [3]:
def training_example(row):
    """Строит пример spaCy только при точном совпадении всех границ с токенами."""
    doc = nlp.make_doc(row["source_text"])
    entities = gold_spans(row)
    if any(doc.char_span(start, end, alignment_mode="strict") is None
           for start, end, _ in entities):
        return None
    return Example.from_dict(doc, {"entities": entities})


usable_train = []
skipped = Counter()
for row in train_rows:
    example = training_example(row)
    if example is None:
        skipped["unaligned"] += 1
    else:
        usable_train.append(row)
print(f"Пригодно для обучения: {len(usable_train)}; исключено: {skipped['unaligned']}")

Пригодно для обучения: 56998; исключено: 1189


In [4]:
def score_rows(pipeline, sample):
    """Считает точные совпадения символьных границ и меток на всей части данных."""
    counts = Counter()
    per_label = {label: Counter() for label in labels}
    docs = pipeline.pipe((row["source_text"] for row in sample), batch_size=64)
    for row, doc in zip(sample, docs, strict=True):
        gold = set(gold_spans(row))
        predicted = {(ent.start_char, ent.end_char, ent.label_) for ent in doc.ents}
        for label in per_label:
            g = {span for span in gold if span[2] == label}
            p = {span for span in predicted if span[2] == label}
            per_label[label].update(gold=len(g), pred=len(p), tp=len(g & p))
    for item in per_label.values():
        counts.update(item)
    return counts, per_label


def prf(counts):
    """Возвращает precision, recall и F1 с нулём при отсутствии предсказаний."""
    precision = counts["tp"] / counts["pred"] if counts["pred"] else 0.0
    recall = counts["tp"] / counts["gold"] if counts["gold"] else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1


def report(part, overall, per_label):
    """Печатает качество по каждой метке и сводную микрооценку."""
    print(f"{part}: gold={overall['gold']}, pred={overall['pred']}, TP={overall['tp']}; P/R/F1={prf(overall)}")
    for label, counts in sorted(per_label.items()):
        print(f"  {label}: gold={counts['gold']}, pred={counts['pred']}, TP={counts['tp']}; P/R/F1={prf(counts)}")
    compatible = per_label["PER"] + per_label["LOC"]
    print(f"  PER+LOC (для сравнения с базовым тестом): {prf(compatible)}")

## Обучение и выбор лучшей эпохи

Только `train` обновляет веса. После каждой эпохи считаем строгую микро-F1 на `val` и сохраняем лучший вариант в `data/models/ru_core_news_sm_pii` (каталог `data/` исключён из Git). Число эпох и размер пакета заданы явно; seed фиксирует порядок примеров. Тестовая часть не участвует в выборе эпохи.

In [5]:
EPOCHS = 5
BATCH_SIZE = 32
DROPOUT = 0.2
SEED = SPLIT["seed"]
MODEL_DIR = ROOT / "data/models/ru_core_news_sm_pii"
random.seed(SEED)
spacy.util.fix_random_seed(SEED)
disabled = [name for name in nlp.pipe_names if name not in {"tok2vec", "ner"}]
best_f1 = -1.0

with nlp.select_pipes(disable=disabled):
    optimizer = nlp.resume_training()
    for epoch in range(EPOCHS):
        random.shuffle(usable_train)
        losses = {}
        for batch_rows in minibatch(usable_train, size=BATCH_SIZE):
            examples = [training_example(row) for row in batch_rows]
            nlp.update(examples, sgd=optimizer, drop=DROPOUT, losses=losses)
        val_total, val_by_label = score_rows(nlp, val_rows)
        val_f1 = prf(val_total)[2]
        print(f"Эпоха {epoch + 1}/{EPOCHS}: loss={losses.get('ner', 0):.1f}, val F1={val_f1:.4f}")
        if val_f1 > best_f1:
            best_f1 = val_f1
            MODEL_DIR.mkdir(parents=True, exist_ok=True)
            nlp.to_disk(MODEL_DIR)
            best_epoch = epoch + 1
    print(f"Лучшая эпоха: {best_epoch}; val F1={best_f1:.4f}; модель: {MODEL_DIR}")
    report("Validation последней эпохи", val_total, val_by_label)

Эпоха 1/5: loss=14027.8, val F1=0.9896
Эпоха 2/5: loss=1685.4, val F1=0.9894
Эпоха 3/5: loss=1057.3, val F1=0.9909
Эпоха 4/5: loss=939.9, val F1=0.9903
Эпоха 5/5: loss=754.8, val F1=0.9909
Лучшая эпоха: 5; val F1=0.9909; модель: /home/aprikhodko/projects/spacy_up/data/models/ru_core_news_sm_pii
Validation последней эпохи: gold=18724, pred=18738, TP=18561; P/R/F1=(0.9905539545308998, 0.9912945951719718, 0.9909241364582778)
  ACCOUNTNUM: gold=1312, pred=1312, TP=1312; P/R/F1=(1.0, 1.0, 1.0)
  BUILDINGNUM: gold=901, pred=893, TP=893; P/R/F1=(1.0, 0.9911209766925638, 0.9955406911928651)
  CREDITCARDNUMBER: gold=729, pred=733, TP=727; P/R/F1=(0.9918144611186903, 0.9972565157750343, 0.9945280437756499)
  DATEOFBIRTH: gold=924, pred=925, TP=924; P/R/F1=(0.9989189189189189, 1.0, 0.9994591671173607)
  DRIVERLICENSENUM: gold=533, pred=533, TP=526; P/R/F1=(0.9868667917448405, 0.9868667917448405, 0.9868667917448405)
  EMAIL: gold=1776, pred=1777, TP=1776; P/R/F1=(0.9994372537985369, 1.0, 0.9997185

## Итоговая оценка

Загружаем сохранённую лучшую модель и однократно оцениваем закреплённый `test`. Для честного сравнения с базовым ноутбуком смотрите строку `PER+LOC`; общая F1 дополнительно охватывает другие метки PII.

In [6]:
best_nlp = spacy.load(MODEL_DIR)
with best_nlp.select_pipes(disable=[name for name in best_nlp.pipe_names if name not in {"tok2vec", "ner"}]):
    val_total, val_by_label = score_rows(best_nlp, val_rows)
    test_total, test_by_label = score_rows(best_nlp, test_rows)
report("Validation лучшей модели", val_total, val_by_label)
report("Test", test_total, test_by_label)

Validation лучшей модели: gold=18724, pred=18738, TP=18561; P/R/F1=(0.9905539545308998, 0.9912945951719718, 0.9909241364582778)
  ACCOUNTNUM: gold=1312, pred=1312, TP=1312; P/R/F1=(1.0, 1.0, 1.0)
  BUILDINGNUM: gold=901, pred=893, TP=893; P/R/F1=(1.0, 0.9911209766925638, 0.9955406911928651)
  CREDITCARDNUMBER: gold=729, pred=733, TP=727; P/R/F1=(0.9918144611186903, 0.9972565157750343, 0.9945280437756499)
  DATEOFBIRTH: gold=924, pred=925, TP=924; P/R/F1=(0.9989189189189189, 1.0, 0.9994591671173607)
  DRIVERLICENSENUM: gold=533, pred=533, TP=526; P/R/F1=(0.9868667917448405, 0.9868667917448405, 0.9868667917448405)
  EMAIL: gold=1776, pred=1777, TP=1776; P/R/F1=(0.9994372537985369, 1.0, 0.9997185477061639)
  IDCARDNUM: gold=806, pred=806, TP=799; P/R/F1=(0.9913151364764268, 0.9913151364764268, 0.9913151364764268)
  LOC: gold=2511, pred=2509, TP=2507; P/R/F1=(0.9992028696691909, 0.9984070091596974, 0.998804780876494)
  PASSWORD: gold=779, pred=788, TP=665; P/R/F1=(0.8439086294416244, 0.853

## Отчёт по классам и матрица ошибок

`sklearn.metrics.classification_report` рассчитан **по сущностям**, а не по отдельным токенам. Сопоставляем эталон и прогноз по одинаковым символьным границам. Если границы совпали, но метка неверна, это ошибка классификации. Если сущность пропущена или найдена лишняя (в том числе с другими границами), используется служебная метка `NONE`: для ошибки границ появляются и пропуск, и лишнее выделение. `NONE` показана в матрице, но исключена из отчёта по целевым классам. Строки матрицы — истинные метки, столбцы — предсказанные. В отчёт входят те же 15 меток, что в микро-F1 выше.

In [1]:
import hashlib
import json
from pathlib import Path

import pandas as pd
import spacy
from IPython.display import display
from sklearn.metrics import classification_report, confusion_matrix

# Ячейка самостоятельно загружает закреплённый тест и обученную модель.
report_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / "notebooks/testing/russian_pii_66k_test_manifest.json").is_file()), None)
if report_root is None:
    raise FileNotFoundError("Не найден корень проекта")
report_manifest = json.loads((report_root / "notebooks/testing/russian_pii_66k_test_manifest.json")
                             .read_text(encoding="utf-8"))
report_data_path = report_root / "data/russian_pii_66k.jsonl"
report_model_path = report_root / "data/models/ru_core_news_sm_pii"
if not report_model_path.is_dir():
    raise FileNotFoundError(f"Не найдена обученная модель: {report_model_path}")
if hashlib.sha256(report_data_path.read_bytes()).hexdigest() != report_manifest["sha256"]:
    raise ValueError("SHA-256 датасета не совпадает с тестовым манифестом")
report_test_ids = set(report_manifest["indices"])
report_test_rows = []
with report_data_path.open(encoding="utf-8") as stream:
    for index, line in enumerate(stream):
        if index in report_test_ids:
            report_test_rows.append((index, json.loads(line)))
if len(report_test_rows) != report_manifest["sample_size"] or index + 1 != report_manifest["row_count"]:
    raise ValueError("Размер тестовой выборки или датасета не совпадает с манифестом")
report_test_rows = [row for _, row in sorted(report_test_rows)]
report_nlp = spacy.load(report_model_path)
report_label_map = {"GIVENNAME": "PER", "SURNAME": "PER", "CITY": "LOC", "STREET": "LOC"}
report_labels = sorted({report_label_map.get(item["label"], item["label"])
                        for row in report_test_rows for item in row["privacy_mask"]})


def report_gold_spans(row):
    """Приводит исходную разметку к меткам модели и объединяет соседние части имени."""
    source = row["source_text"]
    spans = sorted((item["start"], item["end"],
                    report_label_map.get(item["label"], item["label"]))
                   for item in row["privacy_mask"])
    merged = []
    for start, end, label in spans:
        if (label == "PER" and merged and merged[-1][2] == "PER"
                and 0 < start - merged[-1][1] <= 3
                and source[merged[-1][1]:start].isspace()):
            merged[-1] = (merged[-1][0], end, label)
        else:
            merged.append((start, end, label))
    return merged


def report_entity_pairs(pipeline, sample, target_labels):
    """Сопоставляет сущности по границам; отсутствие сущности обозначает как NONE."""
    actual, predicted = [], []
    allowed = set(target_labels)
    docs = pipeline.pipe((row["source_text"] for row in sample), batch_size=64)
    for row, doc in zip(sample, docs, strict=True):
        gold_by_span = {(start, end): label for start, end, label in report_gold_spans(row)}
        pred_by_span = {(ent.start_char, ent.end_char): ent.label_
                        for ent in doc.ents if ent.label_ in allowed}
        for span in sorted(gold_by_span.keys() | pred_by_span.keys()):
            actual.append(gold_by_span.get(span, "NONE"))
            predicted.append(pred_by_span.get(span, "NONE"))
    return actual, predicted


with report_nlp.select_pipes(disable=[name for name in report_nlp.pipe_names
                                     if name not in {"tok2vec", "ner"}]):
    y_true, y_pred = report_entity_pairs(report_nlp, report_test_rows, report_labels)

print(classification_report(y_true, y_pred, labels=report_labels, zero_division=0, digits=4))
matrix_labels = report_labels + ["NONE"]
matrix = confusion_matrix(y_true, y_pred, labels=matrix_labels)
display(pd.DataFrame(matrix, index=pd.Index(matrix_labels, name="Истина"),
                     columns=pd.Index(matrix_labels, name="Прогноз")))

                  precision    recall  f1-score   support

      ACCOUNTNUM     0.9951    1.0000    0.9975       202
     BUILDINGNUM     1.0000    1.0000    1.0000       140
CREDITCARDNUMBER     1.0000    0.9832    0.9915       119
     DATEOFBIRTH     1.0000    1.0000    1.0000       149
DRIVERLICENSENUM     0.9785    0.9891    0.9838        92
           EMAIL     1.0000    1.0000    1.0000       271
       IDCARDNUM     0.9924    0.9848    0.9886       132
             LOC     1.0000    1.0000    1.0000       364
        PASSWORD     0.8633    0.8759    0.8696       137
             PER     0.9955    0.9955    0.9955       445
       SOCIALNUM     1.0000    1.0000    1.0000       117
          TAXNUM     0.9937    1.0000    0.9968       158
    TELEPHONENUM     0.9961    1.0000    0.9980       253
        USERNAME     0.9955    0.9955    0.9955       220
         ZIPCODE     0.9895    0.9895    0.9895        95

       micro avg     0.9900    0.9910    0.9905      2894
       macro

Прогноз,ACCOUNTNUM,BUILDINGNUM,CREDITCARDNUMBER,DATEOFBIRTH,DRIVERLICENSENUM,EMAIL,IDCARDNUM,LOC,PASSWORD,PER,SOCIALNUM,TAXNUM,TELEPHONENUM,USERNAME,ZIPCODE,NONE
Истина,,,,,,,,,,,,,,,,
ACCOUNTNUM,202,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
BUILDINGNUM,0,140,0,0,0,0,0,0,0,0,0,0,0,0,0,0
CREDITCARDNUMBER,0,0,117,0,0,0,0,0,0,0,0,0,1,0,1,0
DATEOFBIRTH,0,0,0,149,0,0,0,0,0,0,0,0,0,0,0,0
DRIVERLICENSENUM,0,0,0,0,91,0,1,0,0,0,0,0,0,0,0,0
EMAIL,0,0,0,0,0,271,0,0,0,0,0,0,0,0,0,0
IDCARDNUM,0,0,0,0,2,0,130,0,0,0,0,0,0,0,0,0
LOC,0,0,0,0,0,0,0,364,0,0,0,0,0,0,0,0
PASSWORD,0,0,0,0,0,0,0,0,120,0,0,0,0,0,0,17
